# Feature Engineering and Ridge Baseline

This notebook uses the prepared player-season datasets to test simple feature choices with a Ridge regression baseline.

Reusable data loading, feature creation, model construction, and temporal evaluation now live in `src/prem_valuation/`. The notebook focuses on the feature decisions and the Ridge baseline result.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd

from prem_valuation.data import load_interim_tables
from prem_valuation.features import (
    ORIGINAL_CATEGORICAL_FEATURES,
    ORIGINAL_NUMERIC_FEATURES,
    RATE_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    SELECTED_FEATURES,
    SELECTED_NUMERIC_FEATURES,
    TARGET,
    add_engineered_features,
)
from prem_valuation.modeling import (
    evaluate_predictions,
    make_ridge_model,
    predict_non_negative,
    walk_forward_evaluate,
)

## Load prepared data

In [ ]:
model_data, scoring_data = load_interim_tables()

model_data = add_engineered_features(model_data)
scoring_data = add_engineered_features(scoring_data)

model_data.shape, scoring_data.shape

## Chronological split

- Train: 2016/17 to 2022/23
- Validation: 2023/24
- Final untouched test: 2024/25
- Unlabelled scoring population: 2025/26

In [ ]:
train_data = model_data.loc[model_data["season"].between(2016, 2022)].copy()
validation_data = model_data.loc[model_data["season"].eq(2023)].copy()
test_data = model_data.loc[model_data["season"].eq(2024)].copy()

for name, dataset in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
    "scoring": scoring_data,
}.items():
    print(name, dataset.shape, sorted(dataset["season"].unique()))

## Feature experiments

Each experiment uses expanding-window validation folds. MAE is the primary metric; RMSE and R2 are diagnostics.

In [ ]:
original_builder = lambda: make_ridge_model(
    ORIGINAL_NUMERIC_FEATURES,
    ORIGINAL_CATEGORICAL_FEATURES,
)
age_curve_builder = lambda: make_ridge_model(
    SELECTED_NUMERIC_FEATURES,
    ORIGINAL_CATEGORICAL_FEATURES,
)
selected_builder = lambda: make_ridge_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)
rates_builder = lambda: make_ridge_model(
    RATE_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)

original_cv = walk_forward_evaluate(
    model_data,
    ORIGINAL_NUMERIC_FEATURES + ORIGINAL_CATEGORICAL_FEATURES,
    TARGET,
    original_builder,
)
age_cv = walk_forward_evaluate(
    model_data,
    SELECTED_NUMERIC_FEATURES + ORIGINAL_CATEGORICAL_FEATURES,
    TARGET,
    age_curve_builder,
)
selected_cv = walk_forward_evaluate(
    model_data,
    SELECTED_FEATURES,
    TARGET,
    selected_builder,
)
rates_cv = walk_forward_evaluate(
    model_data,
    RATE_NUMERIC_FEATURES + SELECTED_CATEGORICAL_FEATURES,
    TARGET,
    rates_builder,
)

cv_summary = pd.DataFrame({
    "original": original_cv[["mae", "rmse", "r2"]].mean(),
    "age_curve": age_cv[["mae", "rmse", "r2"]].mean(),
    "age_and_subposition": selected_cv[["mae", "rmse", "r2"]].mean(),
    "plus_stable_rates": rates_cv[["mae", "rmse", "r2"]].mean(),
}).T

cv_summary

## Ridge alpha tuning

Tune only the selected feature set. The improvement is marginal, but alpha 100 produces the lowest mean walk-forward MAE.

In [ ]:
alpha_values = [0.01, 0.1, 1, 10, 100, 1000]
alpha_results = []

for alpha in alpha_values:
    alpha_builder = lambda alpha=alpha: make_ridge_model(
        SELECTED_NUMERIC_FEATURES,
        SELECTED_CATEGORICAL_FEATURES,
        alpha=alpha,
    )
    fold_results = walk_forward_evaluate(
        model_data,
        SELECTED_FEATURES,
        TARGET,
        alpha_builder,
    )
    alpha_results.append({
        "alpha": alpha,
        "mae": fold_results["mae"].mean(),
        "rmse": fold_results["rmse"].mean(),
        "r2": fold_results["r2"].mean(),
    })

alpha_summary = (
    pd.DataFrame(alpha_results)
    .sort_values("mae")
    .reset_index(drop=True)
)

alpha_summary

In [ ]:
alpha_1_builder = lambda: make_ridge_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    alpha=1,
)
alpha_100_builder = lambda: make_ridge_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    alpha=100,
)

alpha_1_folds = walk_forward_evaluate(
    model_data,
    SELECTED_FEATURES,
    TARGET,
    alpha_1_builder,
)
alpha_100_folds = walk_forward_evaluate(
    model_data,
    SELECTED_FEATURES,
    TARGET,
    alpha_100_builder,
)

alpha_fold_comparison = alpha_1_folds[
    ["validation_season", "mae", "rmse", "r2"]
].merge(
    alpha_100_folds[["validation_season", "mae", "rmse", "r2"]],
    on="validation_season",
    suffixes=("_alpha_1", "_alpha_100"),
)
alpha_fold_comparison["mae_improvement"] = (
    alpha_fold_comparison["mae_alpha_1"]
    - alpha_fold_comparison["mae_alpha_100"]
)

alpha_fold_comparison

## Final Ridge evaluation

Fit the selected Ridge baseline on all development seasons and evaluate once on 2024/25.

In [ ]:
development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True,
)

final_model = make_ridge_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    alpha=100,
)
final_model.fit(
    development_data[SELECTED_FEATURES],
    development_data[TARGET],
)

test_predictions = predict_non_negative(
    final_model,
    test_data[SELECTED_FEATURES],
)

pd.Series(
    evaluate_predictions(test_data[TARGET], test_predictions),
    name="2024/25 final test",
)

## Test residual analysis

In [ ]:
test_results = test_data[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        TARGET,
    ]
].copy()

test_results["predicted_value"] = test_predictions
test_results["residual"] = test_results[TARGET] - test_results["predicted_value"]
test_results["absolute_error"] = test_results["residual"].abs()

test_results.sort_values("absolute_error", ascending=False).head(15)

In [ ]:
test_results["value_band"] = pd.cut(
    test_results[TARGET],
    bins=[0, 5_000_000, 15_000_000, 30_000_000, 60_000_000, float("inf")],
    labels=["EUR 0-5m", "EUR 5-15m", "EUR 15-30m", "EUR 30-60m", "EUR 60m+"],
)

test_results.groupby("value_band", observed=False).agg(
    players=("player_id", "count"),
    actual_mean=(TARGET, "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
)

In [ ]:
test_results.groupby("position").agg(
    players=("player_id", "count"),
    actual_mean=(TARGET, "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
).sort_values("mae", ascending=False)

## Experiment log

- Raw-target Ridge beat the earlier log-target experiment, so later Ridge work used raw market values.
- Nonlinear age helped because values peak around the mid-20s rather than changing linearly.
- Sub-position improved validation because broad positions hide role differences.
- Stable per-90 rates were rejected because their gains were tiny and inconsistent across folds.
- Alpha 100 was selected for the Ridge baseline because it produced the lowest mean walk-forward MAE, though the gain was marginal.
- Final Ridge test performance was about EUR 10.39m MAE and 0.426 R2.
- Residuals showed value compression: low-value players were often overpredicted and elite players were systematically underpredicted.